# Build the SBIR/STTR Firm Outcomes Database

Set the variables below, then use Runtime > Run all. Each step skips work that is already done, so the notebook can be rerun after an interruption.

The repository folder (`sbir-firm-outcomes`) should sit on Google Drive. The data folder can be anywhere on Drive and will be created if it does not exist. `SAM_DIRS` points to folders that already hold SAM extracts, which are copied into the data folder. USPTO files go in `data/raw/patents` (assignment files in `data/raw/patents/assignment`), so `PATENTS_DIRS` is normally left empty.

In [ ]:
REPO_DIR = "/content/drive/MyDrive/SBIR_Firm_Outcomes_Pub/sbir-firm-outcomes"
DATA_DIR = "/content/drive/MyDrive/SBIR_Firm_Outcomes_Pub/data"
SEC_USER_AGENT = "Your Name your.email@institution.edu"
SAM_DIRS = ["/content/drive/MyDrive/sbir_counterfactual/Data_Processing_Scripts"]
PATENTS_DIRS = []   # extra folders to search for USPTO files, if any

# Which steps to run. None runs every step (each skips work already done). For a partial rerun
# list the steps, for example ["places", "outcomes", "panel", "validate", "release"].
STEPS = None

# Census API key, read from Colab Secrets (key icon in the left sidebar, secret named CENSUS_KEY,
# notebook access on) so it never appears in the notebook. Leave the secret unset to go without.
try:
    from google.colab import userdata
    CENSUS_KEY = userdata.get("CENSUS_KEY")
except Exception:
    CENSUS_KEY = ""
CENSUS_KEY = CENSUS_KEY.strip() if isinstance(CENSUS_KEY, str) else ""
print("Census key loaded:", bool(CENSUS_KEY))

In [ ]:
import subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")

In [ ]:
from sbirfo.cli import main

args = ["--data-dir", str(DATA_DIR), "--sec-user-agent", str(SEC_USER_AGENT), "--no-patent-download",
        "--sam-dir", *SAM_DIRS, "--patents-dir", *PATENTS_DIRS]
if CENSUS_KEY:
    args += ["--census-key", CENSUS_KEY]
if STEPS:
    args += ["--steps", *STEPS]
_ = main(args)

To rerun part of the pipeline, pass `--steps`, for example

```python
main(["--data-dir", DATA_DIR, "--steps", "federal", "sam", "patents", "outcomes", "validate", "release",
      "--sam-dir", *SAM_DIRS, "--patents-dir", *PATENTS_DIRS])
```

For the sensitivity version built from medium- as well as high-confidence links, add `"--include-medium-links"`. The USAspending step makes several thousand requests on its first run and saves them as it goes. Phase III contracts are identified from award descriptions; add `"--research-codes"` to also read FPDS research codes through the USAspending download service, which is slow because the service accepts only a few jobs per client.

# Running the code below to validate:

In [ ]:
import zipfile, json
z = zipfile.ZipFile(f"{DATA_DIR}/release/sbir_firm_outcomes_v1.2.0.zip")
readme = z.read("sbir_firm_outcomes_v1.2.0/README.md").decode()
meta = json.loads(z.read("sbir_firm_outcomes_v1.2.0/.zenodo.json"))
print("Validation section:", "## Validation of the SEC linkage" in readme)
print("Results file:", "sbir_firm_outcomes_v1.2.0/validation_results.csv" in z.namelist())
print("Creator:", meta["creators"])